# Installer Osmocom jusqu'au DSP Calypso, étape par étape

Brouillon du 2026-10-03, **non exécuté**. Ce notebook rejoue, dans l'ordre, les étapes du
`Dockerfile` d'osmo-operator (graphe des stages : `Dockerfile:36-83`) sur une machine
Ubuntu 24.04 **native**, jusqu'à un appel vérifié en mode DSP (c54x_exe + mask-ROM TI).

Règles :

- **une étape = une cellule**, suivie de sa cellule de **vérification** ;
- chaque cellule cite en commentaire la ligne du Dockerfile ou du script d'origine ;
- `osmo-deb pack … make install` du Dockerfile (cache `.deb`, `packaging/osmo-deb.sh`) est remplacé
  ici par `sudo make install` : même résultat dans `/usr/local`, sans le paquet ;
- ce qui n'a pas pu être vérifié est marqué **À CONFIRMER**.

Prérequis : Ubuntu 24.04 (base du Dockerfile, `Dockerfile:85`), sudo, ~10 Go libres, réseau
(gitea.osmocom.org, github.com, ftp.freecalypso.org). Racine `/opt/GSM` (`ARG ROOT=/opt/GSM`, `Dockerfile:90`).

Le noyau Jupyter doit tourner sous un utilisateur sudoer. Les cellules `%%bash` ne partagent pas
leur environnement : chaque cellule repose ce dont elle a besoin.

## 0. Racine et variables d'environnement

`Dockerfile:90-95` : `ROOT=/opt/GSM`, `PKG_CONFIG_PATH=/usr/local/lib/pkgconfig`,
`LD_LIBRARY_PATH=/usr/local/lib`, `GIT_TERMINAL_PROMPT=0`. On les écrit dans un fichier sourcé par
les cellules suivantes.

In [ ]:
%%bash
# Dockerfile:90-95 (ARG ROOT, ENV PKG_CONFIG_PATH / LD_LIBRARY_PATH / GIT_TERMINAL_PROMPT)
sudo mkdir -p /opt/GSM && sudo chown "$USER" /opt/GSM
cat > /opt/GSM/.notebook.env <<'EOF'
export ROOT=/opt/GSM
export PKG_CONFIG_PATH=/usr/local/lib/pkgconfig
export LD_LIBRARY_PATH=/usr/local/lib
export GIT_TERMINAL_PROMPT=0
EOF
cat /opt/GSM/.notebook.env

In [ ]:
%%bash
# Verification
. /opt/GSM/.notebook.env && test -w "$ROOT" && echo "OK : $ROOT inscriptible" && lsb_release -ds

## 1. Dépendances système

Liste **unique** du Dockerfile, `Dockerfile:156-205` (le Dockerfile passe par `apt-fast` ; `apt-get`
fait la même chose, plus lentement — `Dockerfile:97-115`). La ligne `debconf` répond d'avance à la
question de wireshark-common (`Dockerfile:143`). Les deb-src sont activés pour le `build-dep gnuradio`
(`Dockerfile:150-155`) : **optionnel** ici, GNU Radio ne sert ni au pont ni au DSP (cellule 1b).

In [ ]:
%%bash
# Dockerfile:143
echo 'wireshark-common wireshark-common/install-setuid boolean true' | sudo debconf-set-selections
# Dockerfile:156-205 (liste recopiee telle quelle, sans le build-dep gnuradio)
sudo apt-get update && sudo apt-get install -y --no-install-recommends \
    build-essential git gcc g++ make cmake autoconf automake libtool pkg-config wget curl \
    libtalloc-dev libpcsclite-dev libsctp-dev libmnl-dev liburing-dev 'asterisk-moh*' \
    libdbi-dev libdbd-sqlite3 libsqlite3-dev sqlite3 libc-ares-dev libgnutls28-dev \
    libortp-dev libfftw3-dev libusb-1.0-0-dev libsofia-sip-ua-dev libsofia-sip-ua-glib-dev \
    python3 python3-dev python3-scapy ca-certificates tmux systemd systemd-sysv dbus-x11 bash-completion \
    gdb-multiarch \
    libasound2-dev libasound2t64 alsa-utils \
    libgsm1-dev libgsm1 \
    iptables iproute2 asterisk ffmpeg \
    psmisc pulseaudio pulseaudio-utils binutils-arm-none-eabi \
    gcc-9 g++-9 gcc-11 g++-11 \
    liblog4cpp5-dev \
    python3-venv python3-pip python3-numpy python3-scipy \
    libglib2.0-dev libpixman-1-dev libslirp-dev socat ninja-build \
    telnet nano whiptail xz-utils libasound2-plugins \
    tcpdump tshark libcap2-bin

In [ ]:
%%bash
# Verification (memes paquets temoins que install_modules/10-deps.sh:80-85)
for p in build-essential libtalloc-dev libsctp-dev libdbi-dev libglib2.0-dev libpixman-1-dev ninja-build python3-numpy socat; do
  dpkg -s "$p" >/dev/null 2>&1 && echo "ok   $p" || echo "MANQUE $p"
done

### 1b. (optionnel) dépendances GNU Radio

Seulement pour gr-gsm / `si_bridge.py` / la FFT (`Dockerfile:785-793`). **Inutile au téléphone émulé**
(le pont n'importe pas GNU Radio : `pont/gsm.py:50,59` charge libosmocoding par ctypes). À CONFIRMER
qu'aucun module de `qosmo/run_modules` ne lance si_bridge.

In [ ]:
%%bash
# Dockerfile:150-155 puis la fin du bloc Dockerfile:203 (apt-fast build-dep -y gnuradio)
# sudo sed -i 's/^Types: deb$/Types: deb deb-src/' /etc/apt/sources.list.d/ubuntu.sources
# sudo apt-get update && sudo apt-get build-dep -y gnuradio
echo "cellule laissee commentee : GNU Radio n'est pas requis pour le DSP"

## 2. Capture non-root et compte de service

`Dockerfile:209` (dumpcap) et `Dockerfile:246` (compte `osmocom`, que les unités systemd Osmocom attendent).

In [ ]:
%%bash
# Dockerfile:209
sudo setcap cap_net_raw,cap_net_admin+eip "$(command -v dumpcap)"
# Dockerfile:246 (identique a install_modules/20-user.sh:6-9)
getent group osmocom  >/dev/null || sudo groupadd osmocom
getent passwd osmocom >/dev/null || sudo useradd -r -g osmocom -s /sbin/nologin -d /var/lib/osmocom osmocom

In [ ]:
%%bash
# Verification
getcap "$(command -v dumpcap)"; getent passwd osmocom

## 3. La pile Osmocom (stage `osmo-core`)

Boucle de `Dockerfile:260-311`, versions et options de configure **identiques**. URL : `osmocom/<nom>`
pour les `libosmo*`, `cellular-infrastructure/<nom>` sinon (`Dockerfile:283-287`). Long (20-30 min).

Pour un téléphone seul, `libosmocore` suffit au DSP et au pont ; le reste est **votre réseau**. Si vous
avez déjà votre pile, ne gardez que `libosmocore:1.12.1` dans la liste (À CONFIRMER : version minimale
de libosmocoding pour `gsm0503_rach_ext_encode`, utilisé par `calypso_bsp.c`, `c54x_exe/Makefile:13`).

`osmo-trx` (patchs IPC, `Dockerfile:326-342`) et les branches gcc-9 d'osmocom-bb (`Dockerfile:603-644`)
ne servent pas au téléphone émulé : non repris.

In [ ]:
%%bash
# Dockerfile:260-311
. /opt/GSM/.notebook.env
set -e
for repo in \
    libosmocore:1.12.1 libosmo-netif:1.7.0 libosmo-abis:2.1.0 libosmo-sigtran:2.2.1 \
    libsmpp34:1.14.5 libgtpnl:1.3.3 osmo-hlr:1.9.2 osmo-mgw:1.15.0 osmo-ggsn:1.14.0 \
    osmo-sgsn:1.13.1 osmo-msc:1.15.0 osmo-bsc:1.14.0 osmo-bts:1.10.0 osmo-pcu:1.5.2 \
    osmo-sip-connector:1.7.2 libosmo-gprs:0.2.1; do
  name=${repo%%:*}; version=${repo##*:}
  case "$name" in libosmo*) GIT_URL="https://gitea.osmocom.org/osmocom/$name" ;;
                  *)        GIT_URL="https://gitea.osmocom.org/cellular-infrastructure/$name" ;; esac
  cd "$ROOT"; [ -d "$name" ] || git clone "$GIT_URL"
  cd "$name"; git checkout "$version"
  autoreconf -fi
  EXTRA_FLAGS=""
  [ "$name" = libosmo-abis ] && EXTRA_FLAGS="--disable-dahdi"
  [ "$name" = osmo-msc ]     && EXTRA_FLAGS="--enable-smpp"
  [ "$name" = osmo-mgw ]     && EXTRA_FLAGS="--enable-alsa"
  [ "$name" = osmo-bts ]     && EXTRA_FLAGS="--enable-virtual --enable-trx"
  [ "$name" = osmo-ggsn ]    && EXTRA_FLAGS="--enable-gtp-linux"
  ./configure $EXTRA_FLAGS
  make -j"$(nproc)"
  sudo make install          # Dockerfile : osmo-deb pack "$name" "$version" make install
  sudo ldconfig
done

In [ ]:
%%bash
# Verification : bibliotheques visibles (c54x_exe/Makefile:14) et demons dans le PATH (install_modules/90-verify.sh:9)
. /opt/GSM/.notebook.env
pkg-config --modversion libosmocore libosmocoding
for b in osmo-stp osmo-hlr osmo-msc osmo-mgw osmo-bsc osmo-bts-trx; do command -v "$b" || echo "MANQUE $b"; done

### 3b. (optionnel) patchs « RAND déterministe » du banc

`Dockerfile:358-414` applique `patches/osmo-{bts,hlr,msc}-force-rand-toy.patch` (authentification
reproductible d'un démarrage à l'autre, `wiki/Home.md`). Ils exigent l'arbre osmo-operator (étape 5).
**À CONFIRMER** : non nécessaires au fonctionnement du DSP. À rejouer *avant* `make install` si on les veut :
`git -C /opt/GSM/osmo-bts apply /opt/GSM/osmo-operator/patches/osmo-bts-force-rand-toy.patch` (idem hlr, msc).

## 4. osmo-gapk (avant osmocom-bb)

`Dockerfile:445-452`. Le mobile du banc utilise `tch-voice io-handler gapk` (`c54x_exe/mobile_pont.cfg`) :
osmocom-bb doit trouver libosmogapk au configure, donc gapk **d'abord**.

In [ ]:
%%bash
# Dockerfile:445-452
. /opt/GSM/.notebook.env && set -e
cd "$ROOT"; [ -d osmo-gapk ] || git clone https://gitea.osmocom.org/osmocom/gapk osmo-gapk
cd osmo-gapk && autoreconf -fi && ./configure --enable-alsa && make -j"$(nproc)" && sudo make install && sudo ldconfig

In [ ]:
%%bash
# Verification
. /opt/GSM/.notebook.env; pkg-config --modversion libosmogapk && command -v osmo-gapk

## 5. osmo-operator (configs, patchs, pont)

Le Dockerfile clone osmo-operator plus tard (`Dockerfile:1059`) ; on en a besoin dès maintenant pour le
patch osmocom-bb, les configs et **le pont** (`pont/`).

In [ ]:
%%bash
# Dockerfile:1059
. /opt/GSM/.notebook.env
cd "$ROOT"; [ -d osmo-operator ] || git clone https://github.com/bbaranoff/osmo-operator

In [ ]:
%%bash
# Verification
ls /opt/GSM/osmo-operator/pont/pont_dsp.py /opt/GSM/osmo-operator/configs/osmo-bts-trx.cfg

## 6. osmocom-bb (stage `bb`) : osmocon, mobile

`Dockerfile:548-558` : clone, patch `clck-gen-frame-tolerance` (fake_trx, `Dockerfile:541-547`),
`cd src && make nofirmware` (outils hôte seulement ; le firmware vient de l'étape 7).
Installation de `mobile` / `trxcon` / `virtphy` : `Dockerfile:1145-1149`.

In [ ]:
%%bash
# Dockerfile:548-558
. /opt/GSM/.notebook.env && set -e
cd "$ROOT"; [ -d osmocom-bb ] || git clone https://gitea.osmocom.org/phone-side/osmocom-bb
git -C osmocom-bb apply --check /opt/GSM/osmo-operator/patches/osmocom-bb-clck-gen-frame-tolerance.patch 2>/dev/null \
  && git -C osmocom-bb apply /opt/GSM/osmo-operator/patches/osmocom-bb-clck-gen-frame-tolerance.patch || true
cd osmocom-bb/src && make nofirmware
# Dockerfile:1145-1149
sudo install -m755 "$ROOT/osmocom-bb/src/host/trxcon/src/trxcon"         /usr/local/bin/trxcon
sudo install -m755 "$ROOT/osmocom-bb/src/host/layer23/src/mobile/mobile" /usr/local/bin/mobile
sudo install -m755 "$ROOT/osmocom-bb/src/host/virt_phy/src/virtphy"      /usr/local/bin/virtphy

In [ ]:
%%bash
# Verification : osmocon reste dans l'arbre (c54x_exe/run.sh:31), mobile dans le PATH
ls -la /opt/GSM/osmocom-bb/src/host/osmocon/osmocon && command -v mobile
ldd "$(command -v mobile)" | grep -i gapk || echo "A CONFIRMER : mobile non lie a libosmogapk"

## 7. Firmware Calypso prébuilt

`Dockerfile:585-588`. `make nofirmware` ne construit pas le firmware : on prend les binaires de
`bbaranoff/firmware`, seul chemin consulté par QEMU (`-kernel .elf`) et osmocon (`.bin`) — `Dockerfile:590-598`.

In [ ]:
%%bash
# Dockerfile:585-588
cd /opt/GSM; [ -d firmware ] || GIT_TERMINAL_PROMPT=0 git clone --depth 1 https://github.com/bbaranoff/firmware /opt/GSM/firmware

In [ ]:
%%bash
# Verification
ls -la /opt/GSM/firmware/board/compal_e88/layer1.highram.elf /opt/GSM/firmware/board/compal_e88/layer1.highram.bin

## 8. qosmo — QEMU Calypso (stage `qemu`)

`Dockerfile:691-705`. Un seul binaire pour les deux montages : `--enable-l1-grgsm`, et
`CALYPSO_DSP_EXTERN=1` au lancement pour le DSP externe (`Dockerfile:676-685`). Long (10-15 min).

In [ ]:
%%bash
# Dockerfile:691-705
. /opt/GSM/.notebook.env && set -e
cd "$ROOT"; [ -d qosmo ] || git clone https://github.com/bbaranoff/qosmO /opt/GSM/qosmo
cd /opt/GSM/qosmo
python3 -m venv ~/.venv-qemu && . ~/.venv-qemu/bin/activate
pip install --no-cache-dir tomli
mkdir -p build && cd build
../configure --target-list=arm-softmmu --enable-l1-grgsm \
    --prefix=/opt/GSM/qemu-install --disable-werror --disable-docs
make -j"$(nproc)"
make install
sudo cp /opt/GSM/qemu-install/bin/qemu-system-arm /usr/local/bin/qemu-system-arm
sudo cp /opt/GSM/qemu-install/bin/qosmo /usr/local/bin/qosmo

In [ ]:
%%bash
# Verification : binaire attendu par c54x_exe/run.sh:28 et qosmo/environnement/bench.env:4
ls -la /opt/GSM/qosmo/build/qemu-system-arm && qosmo -V
/opt/GSM/qosmo/build/qemu-system-arm -M help | grep -i calypso   # A CONFIRMER : libelle exact

## 9. c54x_exe + ROM du DSP (stage `l1`)

`Dockerfile:723-740`. Compile les sources de qosmo, se lie à libosmocoding ; CFLAGS du Dockerfile =
ceux du Makefile **sans `-march=native`** (`Dockerfile:726-728`). La ROM n'est pas dans le dépôt :
`rom/fetch-rom.sh` la télécharge chez FreeCalypso (dump 3606), la convertit et vérifie sa somme.

In [ ]:
%%bash
# Dockerfile:731-740
. /opt/GSM/.notebook.env && set -e
cd "$ROOT"; [ -d c54x_exe ] || git clone https://github.com/bbaranoff/c54x_exe /opt/GSM/c54x_exe
cd /opt/GSM/c54x_exe
make QOSMO=/opt/GSM/qosmo \
  CFLAGS="-O3 -g -Wall -Werror=format -Werror=format-extra-args -Wno-unused-function -Wno-unused-variable -Wno-unused-but-set-variable -Wno-sign-compare"
bash rom/fetch-rom.sh --dest /opt/GSM --dest /opt/GSM/c54x_exe/rom

In [ ]:
%%bash
# Verification : ROM (fetch-rom.sh:35-42 la reverifie) et DSP seul (c54x_exe/README.md:66)
cd /opt/GSM/c54x_exe
bash rom/fetch-rom.sh --dest /opt/GSM     # attendu : « ROM DSP 3606 deja en place et verifiee »
./c54x_exe --trames 200 | tail -5

## 10. (optionnel) grgsm_exe

`Dockerfile:743-747`. Outil de banc (L1 gr-gsm hors QEMU, sans firmware) : **ne fait pas camper le mobile**
(`grgsm_exe/README.md:32-36`). Non requis pour l'appel en DSP.

In [ ]:
%%bash
# Dockerfile:743-747
cd /opt/GSM; [ -d grgsm_exe ] || git clone https://github.com/bbaranoff/grgsm_exE /opt/GSM/grgsm_exe
cd /opt/GSM/grgsm_exe && make QOSMO=/opt/GSM/qosmo && ./grgsm_exe --trames 5000 | tail -5

## 11. Configuration du réseau

Le Dockerfile copie les gabarits de `configs/` dans `/etc/osmocom` (`Dockerfile:1104-1105`, comme
`install_modules/70-configs.sh`, qui n'écrase rien) et la config du mobile (`Dockerfile:1170`).
Les marqueurs `__MCC__`, `__ARFCN__`, `__BSIC__`… sont remplis au lancement par `start-direct.sh`
(`--regen` régénère, `wiki/Home.md`, options) à partir de `globals.conf`
(`generate_configs.sh`, en-tête). Valeurs du banc : MCC 001, MNC 01, DCS1800 (`globals.conf:23-47`).

Pour que le **mobile DSP** trouve la cellule, celle-ci doit être en **ARFCN 514, BSIC 7**
(`mobile_pont.cfg` `stick 514`, `pont/config.py:46-47`) — ou bien changez `PONT_ARFCN`/`PONT_BSIC` et `stick`.

In [ ]:
%%bash
# Dockerfile:1104-1105 / install_modules/70-configs.sh:8,19-20 (cp -n : n'ecrase pas une config existante)
sudo mkdir -p /etc/osmocom /var/log/osmocom /var/run/smsc ~/.osmocom/bb
sudo cp -n /opt/GSM/osmo-operator/configs/*.cfg /etc/osmocom/
cp -n /opt/GSM/osmo-operator/configs/mobile.cfg ~/.osmocom/bb/mobile.cfg
# generate_configs.sh (usage, en-tete) : fixer la cellule attendue par le mobile DSP, puis afficher
cd /opt/GSM/osmo-operator && ./generate_configs.sh --show ARFCN=514 BSIC=7

In [ ]:
%%bash
# Verification : les valeurs effectives, et les gabarits encore a remplir (normal avant --regen)
grep -E '^(MCC|MNC|BAND|ARFCN|BSIC|LAC|ENCRYPTION)=' /opt/GSM/osmo-operator/globals.conf
grep -c '__[A-Z0-9_]*__' /etc/osmocom/osmo-bsc.cfg

## 12. Lancer : le réseau, la BTS, puis le banc DSP

Chemin du banc : `start-direct.sh` (DSP par défaut depuis le 2026-10-01) joue le plan de `qosmo/run.sh`
**sans** `qemu,pty,osmocon,l2` (cœur Osmocom + abonnés HLR + osmo-bts-trx), puis passe la main à
`c54x_exe/run.sh` avec `MODE=dsp PONT=1 INSNS=120000 LOCKSTEP=1 IQ=none CALYPSO_BSP_STREAM=1 CALYPSO_RHEA_DMA_XFER=1`
(bloc « LE BANC DSP » de `start-direct.sh`).

D'abord le plan, sans rien lancer (`wiki/Home.md`, `--list`), puis un essai à blanc (`--dry-run`
imprime les deux commandes transmises, `start-direct.sh` bloc `DRY`). **À CONFIRMER** : `start-direct.sh`
en natif (hors ISO/Docker) exige aussi tmux, asterisk, pulseaudio (installés à l'étape 1).

In [ ]:
%%bash
# wiki/Home.md « Options les plus utiles » ; start-direct.sh --list / --dry-run
cd /opt/GSM/osmo-operator && sudo ./start-direct.sh --list
cd /opt/GSM/osmo-operator && sudo ./start-direct.sh --dry-run

In [ ]:
%%bash
# Lancement reel : --regen remplit les gabarits de /etc/osmocom (wiki/Home.md) ; --no-attach : ne pas prendre le terminal (qosmo/run.sh usage)
cd /opt/GSM/osmo-operator && sudo ./start-direct.sh --regen --no-attach   # --no-attach : start-direct.sh:208 ; --regen : :243

Alternative **manuelle**, processus par processus, avec votre propre pile : voir la page
`Lancer-a-la-main.md` (étapes 0 à 5 : osmo-bts-trx, `c54x_exe --arm`, QEMU, osmocon, mobile, `pont_dsp.py`).

In [ ]:
%%bash
# Verification de la chaine (c54x_exe/run.sh --status, run.sh:239-247) et des lignes cles (LAUNCH.md)
/opt/GSM/c54x_exe/run.sh --status
grep -a "en attente de l'ARM"          /tmp/c54x-pont/dsp.log     | tail -1
grep -a "pont DSP : API RAM partagee"  /tmp/c54x-pont/qemu.log    | tail -1
grep -a "your code is running now"     /tmp/c54x-pont/osmocon.log | tail -1
grep -a "pont TRX : ports"             /tmp/c54x-pont/pont.log    | tail -1

## 13. Vérifier camp, LU, puis un appel

Mêmes contrôles que le banc de tests (`tests/modules/_lib.sh:36-48,83`, `60-camp.sh`, `70-attache.sh`, `90-appel.sh`) :
VTY du mobile DSP = **4347** (`mobile_pont.cfg`), VTY MSC = 4254 (`wiki/Home.md`, ports).

In [ ]:
%%bash
# tests/modules/_lib.sh:36-38 (vty) et 60-camp.sh:10 : attendre « C3 camped normally »
vty() { { printf 'enable\n%s\n' "$2"; sleep 1; } | timeout --foreground 6 nc -N 127.0.0.1 "$1" 2>/dev/null | tr -d '\r'; }
for i in $(seq 1 60); do vty 4347 "show ms" | grep -q "C3 camped normally" && { echo "camp OK apres ${i}x2s"; break; }; sleep 2; done
vty 4347 "show ms" | grep -E "camped|MM|IMSI"

In [ ]:
%%bash
# LU : c54x_exe/run_real.sh (LOCATION UPDATING ACCEPT, lai=) et 70-attache.sh:9-10 (show subscriber cache)
sed 's/\x1b\[[0-9;]*m//g' /tmp/c54x-pont/mobile.log | grep -ac 'LOCATION UPDATING ACCEPT'
sed 's/\x1b\[[0-9;]*m//g' /tmp/c54x-pont/mobile.log | grep -aoE 'lai=[0-9]+-[0-9]+-[0-9]+' | tail -1
{ printf 'enable\nshow subscriber cache\n'; sleep 1; } | timeout 6 nc -N 127.0.0.1 4254 | grep -E 'IMSI|MSISDN'

In [ ]:
%%bash
# Appel : tests/modules/_lib.sh:48 (call 1 <dest>), :83 (call control state), 90-appel.sh:15,22 ; 600 = echo test (wiki/Home.md)
vty() { { printf 'enable\n%s\n' "$2"; sleep 2; } | timeout --foreground 6 nc -N 127.0.0.1 "$1" 2>/dev/null | tr -d '\r'; }
vty 4347 "call 1 600"
for i in $(seq 1 15); do st="$(vty 4347 'show ms' | sed -n 's/^ *call control state: //p' | head -1)"; echo "$st"; [ "$st" = ACTIVE ] && break; sleep 2; done
grep -a "\[montant\] TCH" /tmp/c54x-pont/dsp.log | tail -2     # LAUNCH.md:68-69
vty 4347 "call 1 hangup"

## 14. Arrêter

`./start-direct.sh --stop` arrête la pile **et** le banc DSP, même sans `--dsp` (`start-direct.sh`
« L'ARRÊT NE DEMANDE PAS --dsp ») ; `c54x_exe/run.sh --stop` seul arrête la chaîne téléphone et purge
`/dev/shm/calypso_*` (`run.sh:210-237`).

In [ ]:
%%bash
cd /opt/GSM/osmo-operator && sudo ./start-direct.sh --stop